### Preprocessing code adapted from  
- authors: Peitek Norman, Bergum Annabelle, Rekrut Maurice, Mucke Jonas, Nadig Matthias,Parnin Chris, Siegmund Janet, Apel Sven   
- title: "Correlates of Programmer Efficacy and Their Link to Experience: A Combined EEG and Eye-Tracking Study"  
- version: 1.0.0  
- date-released: 2022-10-01  
- url: "https://github.com/brains-on-code/NoviceVsExpert"  



In [1]:
import os
import glob
import shutil
import mne
from pybv import write_brainvision
import numpy as np
import regex as re
from tqdm.notebook import tqdm

In [3]:
#ica_file= "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_temp/ica/CollectedData_2021-07-29_1_cleaned.fif"
raw_file_in = "C:/Users/Mahima Acharya/Documents/BCI/Data/raw_data/"
raw_set_path = "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_temp/raw_set"
#raw_set_file = "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_temp/raw/CollectedData_2021-07-29_1.set"
raw_ica_path = "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_temp/ica"
save_to = "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_temp/raw_set" 
filepath_out: str = "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_temp/raw"
channels_to_drop = ['x_dir', 'y_dir', 'z_dir']
montage_path= 'AC-64.bvef'



In [4]:
def load_fif_file(filename, filepath):
    full_filename = os.path.join(filepath, filename)
    return mne.io.read_raw_fif(fname=full_filename + '.fif', preload=True)


def convert_fif_to_set(filename, load_from, save_to, number):
    # load fif file
    raw = load_fif_file(filename, load_from)

    # load and set montage
    montage = mne.channels.read_custom_montage(montage_path, head_size=0.085)
    raw.set_montage(montage)

    # drop channels that wont be used
    raw.drop_channels(channels_to_drop)

    # save data to eeglab format
    savepath: str = os.path.join(save_to, filename + "_" + number + ".set")
    mne.export.export_raw(savepath, raw, 'eeglab', 'auto', verbose="ERROR")

    # transform events to eeglab format
    events = raw.info["events"]
    np_e = []
    for e in events:
        np_e.append([e['list'][0], e['list'][2]])

    np_array = np.asarray(np_e)

    # save events to eeglab format
    channels = raw.ch_names
    df = raw.get_data(channels)
    filename = filename + "_" + number
    write_brainvision(data=df, sfreq=500, ch_names=channels, folder_out=filepath_out, fname_base=filename, events=np_array, overwrite=True)

In [5]:

# Use glob to find files ending with ".fif" and replace backslashes with forward slashes
file_paths = [f.replace('\\', '/') for f in glob.glob(os.path.join(raw_file_in, "*.fif"))]

eeg_raw_file_in = "C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_raw_data/"


In [ ]:
#Create folders based on the extracted numbers
for file_path in file_paths:
    # Extract the number from the file name
    number = int(os.path.basename(file_path).split(".")[0][-2:])
    
    # Create a folder with the "participantxx" name
    folder_name = f"participant{number:02}"
    destination_folder  = os.path.join(eeg_raw_file_in, folder_name).replace('\\', '/')
    
   # Create the folder if it doesn't exist
    os.makedirs(destination_folder , exist_ok=True)
    
    # Copy the file to the destination folder with the name "raw.fif"
    shutil.copy(file_path, os.path.join(destination_folder, "eeg_raw.fif"))

print("Folders created and files moved.")

In [6]:
# get all folders in data/filteredData
folders = [f.path for f in os.scandir(eeg_raw_file_in) if f.is_dir()]
# extract the numbers
folders = [f.split('/')[-1] for f in folders]
numbers = [int(re.findall(r'\d+', f)[0]) for f in folders]
# numbers to str with leading zeros
numbers = [str(n).zfill(2) for n in numbers]

mne.set_log_level('ERROR')

# iterate over all folders
for number in tqdm(numbers):
    # construct path to folder
    load_from = eeg_raw_file_in + "/Participant" + number
    save_to = "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_temp/raw"
    file_name = "eeg_raw"
    # transform file via side effect
    convert_fif_to_set(file_name, load_from, save_to, number)

  0%|          | 0/39 [00:00<?, ?it/s]